In [ ]:
# Install required packages (auto-skipped if already installed)
import importlib
if importlib.util.find_spec('qiskit') is None:
    !pip install -q qiskit qiskit-aer qiskit-ibm-runtime pylatexenc
else:
    print("\u2713 Packages already installed")

# To run on real quantum hardware, uncomment and fill in your credentials:
# from qiskit_ibm_runtime import QiskitRuntimeService
# QiskitRuntimeService.save_account(
#     token="<your-api-key>",
#     instance="<your-crn>",
#     overwrite=True
# )

*Орієнтовний час виконання: менше хвилини на процесорі Eagle r3 (УВАГА: це лише приблизна оцінка. Фактичний час може відрізнятися.)*

## Передумови

Щоб забезпечити швидші та ефективніші результати, починаючи з 1 березня 2024 року, схеми та спостережувані величини потрібно перетворювати так, щоб вони використовували лише інструкції, які підтримує QPU (квантовий процесор), перед відправкою до примітивів Qiskit Runtime. Такі схеми і спостережувані називаються *схемами та спостережуваними архітектури набору інструкцій* (ISA). Один із поширених способів це зробити — скористатися функцією `generate_preset_pass_manager` транспілятора. Втім, ти можеш обрати й більш ручний підхід.

Наприклад, ти можеш захотіти орієнтуватися на певну підмножину кубітів на конкретному пристрої. Цей покроковий приклад перевіряє продуктивність різних налаштувань транспілятора, виконуючи повний цикл: створення, транспіляцію та відправку схем.

## Вимоги

Перед початком переконайся, що встановлено наступне:

* Qiskit SDK v1.2 або новіше, з підтримкою [візуалізації](https://docs.quantum.ibm.com/api/qiskit/visualization)
* Qiskit Runtime v0.28 або новіше (`pip install qiskit-ibm-runtime`)

## Налаштування

In [1]:
# Create circuit to test transpiler on
from qiskit import QuantumCircuit
from qiskit.transpiler.preset_passmanagers import generate_preset_pass_manager
from qiskit.circuit.library import grover_operator, DiagonalGate

# Use Statevector object to calculate the ideal output
from qiskit.quantum_info import Statevector
from qiskit.visualization import plot_histogram
from qiskit.transpiler import PassManager

from qiskit.circuit.library import XGate
from qiskit.quantum_info import hellinger_fidelity

## Крок 1: Відображення класичних вхідних даних у квантову задачу
Створи невелику схему, яку транспілятор спробує оптимізувати. У цьому прикладі будується схема, що реалізує алгоритм Гровера з оракулом, що позначає стан `111`. Далі симулюємо ідеальний розподіл (той, який ти отримав би, запустивши схему на ідеальному квантовому комп'ютері нескінченну кількість разів), щоб порівняти його з результатами пізніше.

In [2]:
oracle = DiagonalGate([1] * 7 + [-1])
qc = QuantumCircuit(3)
qc.h([0, 1, 2])
qc = qc.compose(grover_operator(oracle))

qc.draw(output="mpl", style="iqp")

<Image src="../docs/images/guides/circuit-transpilation-settings/extracted-outputs/4ac958d4-b9b5-4939-a359-a9edca7ddb6a-0.svg" alt="Output of the previous code cell" />

In [3]:
ideal_distribution = Statevector.from_instruction(qc).probabilities_dict()

plot_histogram(ideal_distribution)

<Image src="../docs/images/guides/circuit-transpilation-settings/extracted-outputs/6313186e-bc40-432e-9ada-8594d6a26d55-0.svg" alt="Output of the previous code cell" />

## Transpile

Next, transpile the circuits for the QPU. You will compare the performance of the transpiler with `optimization_level` set to `0` (lowest) against `3` (highest). The lowest optimization level does the bare minimum needed to get the circuit running on the device; it maps the circuit qubits to the device qubits and adds swap gates to allow all two-qubit operations. The highest optimization level is much smarter and uses lots of tricks to reduce the overall gate count. Since multi-qubit gates have high error rates and qubits decohere over time, the shorter circuits should give better results.

<Admonition type="important">
    This example uses IBM Quantum&reg; hardware, but you can try it on any Qiskit-compatible QPU.  Your results might be different.
</Admonition>

The following cell transpiles `qc` for both values of `optimization_level`, prints the number of two-qubit gates, and adds the transpiled circuits to a list. Some of the transpiler's algorithms are randomized, so it sets a seed for reproducibility.

In [4]:
# Use Qiskit Runtime to run jobs on hardware
from qiskit_ibm_runtime import (
    QiskitRuntimeService,
    SamplerV2 as Sampler,
)

In [5]:
# Select the backend with the fewest number of jobs in the queue
service = QiskitRuntimeService()
backend = service.least_busy(
    operational=True, simulator=False, min_num_qubits=127
)
backend.name

'ibm_marrakesh'

In [6]:
# Need to add measurements to the circuit
qc.measure_all()

# Find the correct two-qubit gate
twoQ_gates = set(["ecr", "cz", "cx"])
for gate in backend.basis_gates:
    if gate in twoQ_gates:
        twoQ_gate = gate

circuits = []
for optimization_level in [0, 3]:
    pm = generate_preset_pass_manager(
        optimization_level, backend=backend, seed_transpiler=0
    )
    t_qc = pm.run(qc)
    print(
        f"Two-qubit gates (optimization_level={optimization_level}): ",
        t_qc.count_ops()[twoQ_gate],
    )
    circuits.append(t_qc)

Two-qubit gates (optimization_level=0):  21
Two-qubit gates (optimization_level=3):  12


Since CNOTs usually have a high error rate, the circuit transpiled with `optimization_level=3` should perform much better.

Another way you can improve performance is through [dynamical decoupling](/docs/api/qiskit/qiskit.transpiler.passes.PadDynamicalDecoupling), by applying a sequence of gates to idling qubits. This cancels out some unwanted interactions with the environment. The following cell adds dynamic decoupling to the circuit transpiled with `optimization_level=3` and adds it to the list.

In [7]:
from qiskit_ibm_runtime.transpiler.passes.scheduling import (
    ASAPScheduleAnalysis,
    PadDynamicalDecoupling,
)

# Get gate durations so the transpiler knows how long each operation takes
durations = backend.target.durations()

# This is the sequence we'll apply to idling qubits
dd_sequence = [XGate(), XGate()]

# Run scheduling and dynamic decoupling passes on circuit
pm = PassManager(
    [
        ASAPScheduleAnalysis(durations),
        PadDynamicalDecoupling(durations, dd_sequence),
    ]
)
circ_dd = pm.run(circuits[1])

# Add this new circuit to our list
circuits.append(circ_dd)

In [8]:
circ_dd.draw(output="mpl", style="iqp", idle_wires=False)

<Image src="../docs/images/guides/circuit-transpilation-settings/extracted-outputs/c1c91fbd-acfe-413e-a6c9-ad97f4dd5543-0.svg" alt="Output of the previous code cell" />

## Execute the circuit

At this point, you have a list of circuits transpiled with different settings. Next, run these circuits using the Sampler primitive and store the results to `result`.

In [9]:
sampler = Sampler(backend)
job = sampler.run(
    [(circuit) for circuit in circuits],  # sample all three circuits
    shots=8000,
)
result = job.result()

![Output of the previous code cell](../docs/images/guides/circuit-transpilation-settings/extracted-outputs/4ada6498-b9d7-4d88-b8a9-ef1dc0a85bf7-0.avif)

## Крок 3: Виконання за допомогою примітивів Qiskit
На цьому етапі у тебе є список схем, транспільованих для вказаного QPU. Далі створи екземпляр примітива sampler і запусти пакетне завдання за допомогою контекстного менеджера (`with ...:`), який автоматично відкриває та закриває пакет.

У межах контекстного менеджера виконай вибірку зі схем і збережи результати в `result`.

In [10]:
binary_prob = [
    {
        k: v / res.data.meas.num_shots
        for k, v in res.data.meas.get_counts().items()
    }
    for res in result
]
plot_histogram(
    binary_prob + [ideal_distribution],
    bar_labels=False,
    legend=[
        "optimization_level=0",
        "optimization_level=3",
        "optimization_level=3 + dd",
        "ideal distribution",
    ],
)

<Image src="../docs/images/guides/circuit-transpilation-settings/extracted-outputs/9e86132d-a8b2-40db-af42-53042dfa108b-0.svg" alt="Output of the previous code cell" />

## Крок 4: Постобробка та повернення результату в бажаному класичному форматі
Нарешті, побудуй графіки результатів, отриманих на пристрої, і порівняй їх з ідеальним розподілом. Видно, що результати з `optimization_level=3` ближчі до ідеального розподілу завдяки меншій кількості вентилів, а `optimization_level=3 + dd` — ще ближчі завдяки динамічному роз'єднанню.

In [11]:
for prob in binary_prob:
    print(f"{hellinger_fidelity(prob, ideal_distribution):.3f}")

0.985
0.989
0.988


![Output of the previous code cell](../docs/images/guides/circuit-transpilation-settings/extracted-outputs/525777ea-d438-4f3b-acb6-53e579f24a0e-0.avif)

Ти можеш підтвердити це, обчисливши [вірність Гелінгера](https://docs.quantum.ibm.com/api/qiskit/quantum_info) між кожним набором результатів та ідеальним розподілом (чим вище, тим краще; 1 — ідеальна вірність).